In [1]:
import os

from tqdm import tqdm
from ase.db import connect

from fairchem.core.datasets import AseDBDataset

W0313 10:58:39.108000 85379 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


# Validation Dataset

## Load Dataset

In [2]:
#!time curl -O "https://dl.fbaipublicfiles.com/opencatalystproject/data/omol/250514/val.tar.gz"
#!tar -xzvf val.tar.gz

In [3]:
dataset_path = "/Volumes/JAC_Backup/OpenFF/Meta-OMol25/val"
output_path = os.path.split(dataset_path)[0]
dataset = AseDBDataset({"src": dataset_path})

## Filter Atoms in ASE Databases

In [4]:
#target_ds_ids = ["elytes", "biomolecules", "metal_complexes", "spice", "orbnet_denali", "geom_orca6", "reactivity", "ani2x", "rgd", "trans1x"]
target_ds_ids = ["geom_orca6"]

os.makedirs(os.path.join(output_path,"ase_databases"), exist_ok=True)
ase_dbs = {ds_name: connect(os.path.join(output_path, f"ase_databases/{ds_name}.db")) for ds_name in target_ds_ids}

In [5]:
for id in tqdm(dataset.ids):
    atoms = dataset.get_atoms(id)
    if atoms.info["data_id"] not in target_ds_ids:
        continue
    atoms.info["omol25-index"] = id
    atoms.info["omol25-split"] = "val"
    ase_dbs[f"{atoms.info['data_id']}"].write(atoms, data=atoms.info)


100%|██████████| 2762021/2762021 [18:33<00:00, 2481.53it/s] 


In [6]:
# Get atoms.info later with:
#    last_rowid = ase_dbs["geom_orca6"].count()   
#    row = ase_dbs["geom_orca6"].get(id=last_rowid)
#    atoms2 = row.toatoms()
#    atoms2.info = row.data

# Training Dataset

In [7]:
#!time curl -O "https://dl.fbaipublicfiles.com/opencatalystproject/data/omol/250514/train.tar.gz"
#!tar -xzvf train.tar.gz

In [8]:
dataset_path_train = "/Volumes/JAC_Backup/OpenFF/Meta-OMol25/train"
output_path = os.path.split(dataset_path)[0]
dataset_train = AseDBDataset({"src": dataset_path_train})

## Filter Atoms in ASE Databases

In [9]:
#target_ds_ids = ["elytes", "biomolecules", "metal_complexes", "spice", "orbnet_denali", "geom_orca6", "reactivity", "ani2x", "rgd", "trans1x"]
target_ds_ids = ["geom_orca6"]

os.makedirs(os.path.join(output_path,"ase_databases"), exist_ok=True)
ase_dbs = {ds_name: connect(os.path.join(output_path, f"ase_databases/{ds_name}.db")) for ds_name in target_ds_ids}

In [10]:
for id in tqdm(dataset_train.ids):
    atoms = dataset_train.get_atoms(id)
    if atoms.info["data_id"] not in target_ds_ids:
        continue
    atoms.info["omol25-index"] = id
    atoms.info["omol25-split"] = "train"
    ase_dbs[f"{atoms.info['data_id']}"].write(atoms, data=atoms.info)

100%|██████████| 101666280/101666280 [78:53:14<00:00, 357.99it/s]    
